# Corrleation Analysis
**Author:** Jakob Balkovec

### Description

I had an idea **What if I collapse all of the families into a single feature?** This notebook explores that idea. It could work but it comes with a tradeoff. If the families are highly correlated, then it's a solid approach, but if not we lose signal.

In [78]:
from pathlib import Path
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from itertools import combinations, product
import shap

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    explained_variance_score
)

TRAIN = Path("/Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/train_derived_new.csv")
VAL = Path("/Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/val_derived_new.csv")
TEST = Path("/Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/test_derived_new.csv")

In [79]:
df_train = pd.read_csv(TRAIN)
df_val = pd.read_csv(VAL)
df_test = pd.read_csv(TEST)

assert len(df_train.columns) == len(df_val.columns) == len(df_test.columns)
print("num_cols: ", len(df_train.columns))

num_cols:  356


In [80]:
df_val = df_val[df_train.columns]
df_test = df_test[df_train.columns]

df_train["split"] = "train"
df_val["split"] = "val"
df_test["split"] = "test"

df_all = pd.concat([df_train, df_val, df_test], axis=0, ignore_index=True)

print("shape:", df_all.shape)

shape: (22720, 357)


In [81]:
target_col = "soil_moisture_5cm"

exclude_cols = {
    target_col,
    "station_id",
    "date",
    "timestamp",
    "split",
    "fold",
    "path",
    "id"
}

feature_cols = [
    c for c in df_all.columns
    if c not in exclude_cols
]

In [82]:
family_rules = {
    "A": lambda c: c.startswith("A_"),
    "B": lambda c: c.startswith("V_"),
    "C": lambda c: c.startswith("C_"),
    "D": lambda c: c.startswith("D_"),
    "E": lambda c: c.startswith("E_"),
    "F": lambda c: c.startswith("F_"),
    "G": lambda c: c.startswith("G_"),
    # "H": lambda c: c.startswith("H_"),
    "I": lambda c: c.startswith("I_"),
}

In [83]:
family_cols = {k: [] for k in family_rules.keys()}

for col in feature_cols:
    for fam, rule in family_rules.items():
        if rule(col):
            family_cols[fam].append(col)

# print summary
for fam, cols in family_cols.items():
    print(f"{fam}: {len(cols)} features")

A: 80 features
B: 168 features
C: 56 features
D: 14 features
E: 7 features
F: 3 features
G: 6 features
I: 1 features


In [84]:
family_dfs = {}

for fam, cols in family_cols.items():
    if len(cols) == 0:
        continue

    family_dfs[fam] = df_all[cols + [target_col]].copy()

print("Created family-specific DataFrames:")
print(list(family_dfs.keys()))

Created family-specific DataFrames:
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'I']


In [85]:
corr_summary = []

for fam, cols in family_cols.items():
    if len(cols) == 0:
        continue

    corrs = df_all[cols].corrwith(df_all[target_col])
    abs_corr = corrs.abs()

    corr_summary.append({
        "family": fam,
        "n_features": len(cols),
        "mean_abs_corr": abs_corr.mean(),
        "max_abs_corr": abs_corr.max(),
        "median_abs_corr": abs_corr.median()
    })

corr_df = pd.DataFrame(corr_summary).sort_values("max_abs_corr", ascending=False)
print(corr_df)

  family  n_features  mean_abs_corr  max_abs_corr  median_abs_corr
1      B         168       0.190030      0.643991         0.140401
2      C          56       0.202081      0.630930         0.144115
3      D          14       0.103132      0.499941         0.047648
6      G           6       0.332817      0.432332         0.395188
4      E           7       0.078021      0.150338         0.100690
0      A          80       0.045355      0.125305         0.033634
7      I           1       0.095705      0.095705         0.095705
5      F           3       0.074738      0.090849         0.072293


In [86]:
results = []

for fam, cols in family_cols.items():
    if len(cols) == 0:
        continue

    X_tr = df_train[cols].values
    X_val = df_val[cols].values
    y_tr = df_train["soil_moisture_5cm"].values
    y_val = df_val["soil_moisture_5cm"].values

    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)

    r2 = r2_score(y_val, pred)
    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    bias = np.mean(pred - y_val)
    pearson = np.corrcoef(y_val, pred)[0, 1] if len(y_val) > 1 else np.nan
    ev = explained_variance_score(y_val, pred)
    nrmse_std = rmse / np.std(y_val)
    p90_abs_err = np.percentile(np.abs(y_val - pred), 90)

    results.append({
        "family": fam,
        "n_features": len(cols),
        "val_r2": r2,
        "mae": mae,
        "rmse": rmse,
        "bias": bias,
        "pearson": pearson,
        "explained_var": ev,
        "nrmse_std": nrmse_std,
        "p90_abs_err": p90_abs_err,
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values("val_r2", ascending=False))

  family  n_features    val_r2       mae      rmse      bias   pearson  \
2      C          56  0.748484  0.038790  0.050512 -0.026580  0.905074   
1      B         168  0.739172  0.040070  0.051438 -0.029199  0.907372   
0      A          80  0.646000  0.047698  0.059925 -0.026363  0.845806   
3      D          14  0.623553  0.046880  0.061796 -0.022438  0.821798   
6      G           6  0.584335  0.051712  0.064935 -0.021699  0.798324   
5      F           3  0.072498  0.075448  0.096998 -0.023015  0.450270   
7      I           1 -0.065352  0.093644  0.103957 -0.027464  0.095381   
4      E           7 -0.169217  0.091945  0.108907 -0.042283  0.276987   

   explained_var  nrmse_std  p90_abs_err  
2       0.818131   0.501514     0.084440  
1       0.823221   0.510713     0.086951  
0       0.714515   0.594979     0.098722  
3       0.673185   0.613552     0.102834  
6       0.630750   0.644721     0.106494  
5       0.124716   0.963069     0.172954  
7       0.009005   1.032159     

---

In [87]:
combo_definitions = {
    "C_only": ["C"],
    "B_only": ["B"],
    "C_B": ["C", "B"],
    "C_D": ["C", "D"],
    "C_B_A": ["C", "B", "A"],
    "C_B_D": ["C", "B", "D"],
    "C_B_G": ["C", "B", "G"],
    "C_B_A_D": ["C", "B", "A", "D"],
    "ALL_strong": ["C", "B", "A", "D", "G"],
}

In [88]:
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    pearson = np.corrcoef(y_true, y_pred)[0, 1]
    explained_var = 1 - np.var(y_true - y_pred) / np.var(y_true)
    nrmse_std = rmse / np.std(y_true)
    p90_abs_err = np.percentile(np.abs(y_true - y_pred), 90)

    return {
        "r2": r2,
        "mae": mae,
        "rmse": rmse,
        "bias": bias,
        "pearson": pearson,
        "explained_var": explained_var,
        "nrmse_std": nrmse_std,
        "p90_abs_err": p90_abs_err,
    }

In [89]:
results = []

for combo_name, fam_list in combo_definitions.items():

    # gather all columns from those families
    cols = []
    for fam in fam_list:
        cols.extend(family_cols.get(fam, []))

    if len(cols) == 0:
        continue

    X_tr = df_train[cols].values
    X_val = df_val[cols].values
    y_tr = df_train["soil_moisture_5cm"].values
    y_val = df_val["soil_moisture_5cm"].values

    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)

    metrics = compute_metrics(y_val, pred)

    results.append({
        "combo": combo_name,
        "families": ",".join(fam_list),
        "n_features": len(cols),
        **metrics
    })

results_df = pd.DataFrame(results).sort_values("r2", ascending=False)
print(results_df.to_string(index=False))

     combo  families  n_features       r2      mae     rmse      bias  pearson  explained_var  nrmse_std  p90_abs_err
ALL_strong C,B,A,D,G         324 0.774415 0.036164 0.047837 -0.024912 0.914434       0.835592   0.474958     0.083093
     C_B_D     C,B,D         238 0.772442 0.036174 0.048046 -0.023740 0.910715       0.827999   0.477030     0.083484
   C_B_A_D   C,B,A,D         318 0.769685 0.036804 0.048336 -0.025117 0.912698       0.831874   0.479912     0.084305
     C_B_G     C,B,G         230 0.762757 0.038233 0.049057 -0.027408 0.914812       0.836807   0.487076     0.083829
     C_B_A     C,B,A         304 0.762338 0.037948 0.049101 -0.027368 0.914748       0.836172   0.487506     0.083110
       C_B       C,B         224 0.749460 0.039107 0.050413 -0.028256 0.910073       0.828166   0.500539     0.083733
       C_D       C,D          70 0.748799 0.038389 0.050480 -0.026058 0.904277       0.815736   0.501199     0.088129
    C_only         C          56 0.748484 0.038790 0.050

**Model behavior:**
```
Soil Moisture(t)
~= Memory(t-1…t-k)
+ Seasonal regime
+ Mild smoothing stabilization
+ Minor forcing refinement
```

In [90]:
all_strong_families = combo_definitions["ALL_strong"]
all_strong_cols = []
for fam in all_strong_families:
    all_strong_cols.extend(family_cols.get(fam, []))
all_strong_cols = list(dict.fromkeys(all_strong_cols))

X_train_all = df_train[all_strong_cols].copy()
X_val_all = df_val[all_strong_cols].copy()
X_test_all = df_test[all_strong_cols].copy()

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

In [91]:
RAIN_MONOTONE_POSITIVE = {"G_rain_sum_3d", "G_rain_sum_7d", "G_rain_sum_30d"}

def build_feature_weights(cols, c_weight=1.35, d_weight=1.35):
    weights = []
    for c in cols:
        w = 1.0
        if c.startswith("C_"):
            w *= c_weight
        if c.startswith("D_"):
            w *= d_weight
        weights.append(w)
    return np.asarray(weights, dtype=float)

def build_monotone_constraints(cols):
    return tuple(1 if c in RAIN_MONOTONE_POSITIVE else 0 for c in cols)

def fit_eval_xgb(X_tr, y_tr, X_va, y_va, X_te, y_te, cols, params, use_priors=True):
    model_kwargs = dict(params)
    if use_priors:
        model_kwargs["feature_weights"] = build_feature_weights(cols)
        model_kwargs["monotone_constraints"] = build_monotone_constraints(cols)

    model = XGBRegressor(**model_kwargs)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    val_pred = model.predict(X_va)
    test_pred = model.predict(X_te)

    return model, compute_metrics(y_va, val_pred), compute_metrics(y_te, test_pred)

In [92]:
baseline_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

baseline_model, baseline_val, baseline_test = fit_eval_xgb(
    X_train_all,
    y_train,
    X_val_all,
    y_val,
    X_test_all,
    y_test,
    all_strong_cols,
    baseline_params,
    use_priors=False,
)

In [93]:
search_space = {
    "max_depth": [6, 7, 8],
    "min_child_weight": [5, 8],
    "reg_alpha": [0.05, 0.2],
    "reg_lambda": [2.0, 6.0],
    "colsample_bytree": [0.75, 0.9],
}

fixed_params = {
    "n_estimators": 2500,
    "learning_rate": 0.03,
    "subsample": 0.85,
    "gamma": 0.1,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

tuning_rows = []
best_model = None
best_params = None
best_val_r2 = -np.inf

In [ ]:
for max_depth, min_child_weight, reg_alpha, reg_lambda, colsample_bytree in product(
    search_space["max_depth"],
    search_space["min_child_weight"],
    search_space["reg_alpha"],
    search_space["reg_lambda"],
    search_space["colsample_bytree"],
):
    trial_params = {
        **fixed_params,
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "colsample_bytree": colsample_bytree,
    }

    model, val_metrics, test_metrics = fit_eval_xgb(
        X_train_all,
        y_train,
        X_val_all,
        y_val,
        X_test_all,
        y_test,
        all_strong_cols,
        trial_params,
        use_priors=True,
    )

    tuning_rows.append(
        {
            **trial_params,
            "val_r2": val_metrics["r2"],
            "val_rmse": val_metrics["rmse"],
            "test_r2": test_metrics["r2"],
            "test_rmse": test_metrics["rmse"],
        }
    )

    if val_metrics["r2"] > best_val_r2:
        best_val_r2 = val_metrics["r2"]
        best_model = model
        best_params = trial_params
        best_val_metrics = val_metrics
        best_test_metrics = test_metrics

tuning_df = pd.DataFrame(tuning_rows).sort_values("val_r2", ascending=False)
print("Top tuned configs (ALL_strong):")
print(tuning_df.head(10).to_string(index=False))

# 20 mins

[18:06:35] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.

[18:06:56] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.

[18:07:18] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.

[18:07:41] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.

[18:08:03] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.

[18:08:24] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.

[18:08:47] WARNING: /Users/runner/miniforge3/conda-b

Top tuned configs (ALL_strong):
 n_estimators  learning_rate  subsample  gamma        objective  random_state  n_jobs tree_method  max_depth  min_child_weight  reg_alpha  reg_lambda  colsample_bytree   val_r2  val_rmse  test_r2  test_rmse
         2500           0.03       0.85    0.1 reg:squarederror            42      -1        hist          8                 8       0.20         6.0              0.90 0.783241  0.046892 0.734424   0.048200
         2500           0.03       0.85    0.1 reg:squarederror            42      -1        hist          8                 5       0.20         6.0              0.75 0.782958  0.046922 0.736697   0.047993
         2500           0.03       0.85    0.1 reg:squarederror            42      -1        hist          6                 8       0.20         6.0              0.90 0.782200  0.047004 0.733018   0.048327
         2500           0.03       0.85    0.1 reg:squarederror            42      -1        hist          6                 5       0.20   

In [95]:
shap_sample_n = min(3000, len(X_train_all))
X_shap = X_train_all.sample(shap_sample_n, random_state=42)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list):
    shap_values = shap_values[0]

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_rank = pd.Series(mean_abs_shap, index=all_strong_cols).sort_values(ascending=False)

PRUNE_BOTTOM_FRAC = 0.25
n_keep = max(12, int(np.ceil(len(all_strong_cols) * (1 - PRUNE_BOTTOM_FRAC))))
selected_cols = shap_rank.head(n_keep).index.tolist()

In [96]:
X_train_pruned = df_train[selected_cols].copy()
X_val_pruned = df_val[selected_cols].copy()
X_test_pruned = df_test[selected_cols].copy()

pruned_model, pruned_val_metrics, pruned_test_metrics = fit_eval_xgb(
    X_train_pruned,
    y_train,
    X_val_pruned,
    y_val,
    X_test_pruned,
    y_test,
    selected_cols,
    best_params,
    use_priors=True,
)

[18:26:38] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1730232611148/work/src/learner.cc:740: 
Parameters: { "feature_weights" } are not used.



In [97]:
print("Top SHAP features (ALL_strong):")
print(shap_rank.head(20).to_string())

summary_df = pd.DataFrame(
    [
        {
            "model": "ALL_strong_baseline_depth6",
            "n_features": len(all_strong_cols),
            "val_r2": baseline_val["r2"],
            "val_rmse": baseline_val["rmse"],
            "test_r2": baseline_test["r2"],
            "test_rmse": baseline_test["rmse"],
        },
        {
            "model": "ALL_strong_tuned_depth6_8_reg_weighted_monotone",
            "n_features": len(all_strong_cols),
            "val_r2": best_val_metrics["r2"],
            "val_rmse": best_val_metrics["rmse"],
            "test_r2": best_test_metrics["r2"],
            "test_rmse": best_test_metrics["rmse"],
        },
        {
            "model": "ALL_strong_shap_pruned_retrain",
            "n_features": len(selected_cols),
            "val_r2": pruned_val_metrics["r2"],
            "val_rmse": pruned_val_metrics["rmse"],
            "test_r2": pruned_test_metrics["r2"],
            "test_rmse": pruned_test_metrics["rmse"],
        },
    ]
).sort_values("val_r2", ascending=False)

print()
print(f"SHAP pruning kept {len(selected_cols)} / {len(all_strong_cols)} features")
print(summary_df.to_string(index=False))

all_strong_best_params = best_params
all_strong_shap_rank = shap_rank
all_strong_selected_cols = selected_cols
all_strong_tuning_table = tuning_df


Top SHAP features (ALL_strong):
V_rollmin_LST_modis_kobs30     0.022717
D_sin_DOY                      0.012369
V_ema_G_API_kobs7              0.010160
V_ema_LST_modis_kobs30         0.010073
V_rollmin_E_SAR_diff_kobs30    0.008051
V_rollmin_G_API_kobs30         0.005363
C_lag_G_API_kobs1              0.005182
V_rollmin_F_NDMI_kobs30        0.005018
V_ema_G_API_kobs14             0.004042
G_rain_sum_3d                  0.003881
D_z_F_NDMI                     0.003576
V_rollmin_F_NDMI_kobs14        0.003411
C_lag_LST_modis_kobs30         0.003210
G_rain_sum_7d                  0.003076
D_sa_F_NDMI                    0.002543
G_API                          0.002413
G_rain_sum_30d                 0.002248
V_ema_F_NDMI_kobs30            0.001741
V_rollmax_E_SAR_diff_kobs14    0.001534
D_z_E_SAR_ratio                0.001529

SHAP pruning kept 243 / 324 features
                                          model  n_features   val_r2  val_rmse  test_r2  test_rmse
ALL_strong_tuned_depth6_8_reg_w

In [ ]:
use_families = ["A", "B", "C", "D", "E", "F", "G"]
cols = []
for fam in use_families:
    cols.extend(family_cols.get(fam, []))

cols = sorted(list(dict.fromkeys(cols)))  # stable unique

feat2idx = {c: i for i, c in enumerate(cols)}

def fam_idx(fam):
    return [feat2idx[c] for c in family_cols.get(fam, []) if c in feat2idx]

interaction_constraints = [
    fam_idx("C") + fam_idx("B"),  # C with B
    fam_idx("C") + fam_idx("D"),  # C with D
    fam_idx("C") + fam_idx("G"),  # C with G
    fam_idx("E") + fam_idx("F"),  # E with F
]

# optional: let A interact only with C (if you want)
# interaction_constraints.append(fam_idx("A") + fam_idx("C"))
interaction_constraints = [g for g in interaction_constraints if len(g) > 0]

print("n_features:", len(cols))
print("n_constraint_groups:", len(interaction_constraints))
print("group sizes:", [len(g) for g in interaction_constraints])

n_features: 334
n_constraint_groups: 4
group sizes: [224, 70, 62, 10]


In [102]:
X_tr = df_train[cols]
y_tr = df_train[target_col].values

X_val = df_val[cols]
y_val = df_val[target_col].values

model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=8,
    subsample=0.85,
    colsample_bytree=0.90,
    gamma=0.1,
    reg_alpha=0.20,
    reg_lambda=6.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    interaction_constraints=interaction_constraints,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [103]:
def predict_split(model, df, cols, target_col):
    X = df[cols].values
    y_true = df[target_col].values
    y_pred = model.predict(X)
    return y_true, y_pred

y_train_true, y_train_pred = predict_split(model, df_train, cols, target_col)
y_val_true, y_val_pred     = predict_split(model, df_val, cols, target_col)
y_test_true, y_test_pred   = predict_split(model, df_test, cols, target_col)

In [104]:
rows = []

for name, yt, yp in [
    ("train", y_train_true, y_train_pred),
    ("val", y_val_true, y_val_pred),
    ("test", y_test_true, y_test_pred),
]:
    m = compute_metrics(yt, yp)
    rows.append({"split": name, **m})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

split       r2      mae     rmse      bias  pearson  explained_var  nrmse_std  p90_abs_err
train 0.902923 0.024064 0.031859 -0.000016 0.952694       0.902923   0.311572     0.050947
  val 0.757707 0.037967 0.049577 -0.027940 0.914501       0.834665   0.492232     0.085476
 test 0.712381 0.039650 0.050160 -0.019399 0.870024       0.755399   0.536301     0.081807
